In [11]:
from nba_api.stats.endpoints import LeagueDashLineups
from nba_api.stats.endpoints import TeamPlayerOnOffDetails
from nba_api.stats.endpoints import LeagueDashPlayerStats
from nba_api.stats.static import teams

import os
import numpy as np
import time
import random
import pandas as pd
from requests.exceptions import ConnectionError, Timeout

In [3]:
FOUR_FACTORS = [
    "EFG_PCT",
    "FTA_RATE",
    "TM_TOV_PCT",
    "OREB_PCT",
    "OPP_EFG_PCT",
    "OPP_FTA_RATE",
    "OPP_TOV_PCT",
    "OPP_OREB_PCT"
]

### Data Gathering

In [4]:
def get_lineups(season, cache_dir="cache"):

    os.makedirs(cache_dir, exist_ok=True)

    path = f"{cache_dir}/lineups_{season}.csv"

    # load cache
    if os.path.exists(path):
        print("Loading cached:", season)
        return pd.read_csv(path)


    retries = 8

    for attempt in range(retries):

        try:

            print(
                f"Downloading {season}, attempt {attempt+1}/{retries}"
            )

            df = LeagueDashLineups(
                season=season,
                season_type_all_star="Regular Season",
                measure_type_detailed_defense="Four Factors",
                per_mode_detailed="Per100Possessions",
                group_quantity=5,
                timeout=60
            ).get_data_frames()[0]


            df.to_csv(path, index=False)

            print("Saved", season)

            time.sleep(random.uniform(2,4))

            return df


        except (ConnectionError, Timeout) as e:

            wait = 10 * (attempt + 1)

            print(
                f"Connection failed. Waiting {wait}s..."
            )

            time.sleep(wait)


    raise Exception(
        f"Could not download lineup data for {season}"
    )

In [5]:
lineups = get_lineups("2021-22")

print(lineups.shape)
print(lineups.head())

Saved 2021-22
(2000, 32)
  GROUP_SET                                  GROUP_ID  \
0   Lineups     -201145-203115-203932-203999-1628420-   
1   Lineups   -101108-203109-1626164-1628969-1629028-   
2   Lineups    -201144-202711-203497-1626220-1628378-   
3   Lineups   -201143-203935-1627759-1628369-1629057-   
4   Lineups  -201976-1626156-1626157-1629020-1630162-   

                                          GROUP_NAME     TEAM_ID  \
0  J. Green - W. Barton - A. Gordon - N. Jokic - ...  1610612743   
1  C. Paul - J. Crowder - D. Booker - M. Bridges ...  1610612756   
2  M. Conley - B. Bogdanovic - R. Gobert - R. O'N...  1610612762   
3  A. Horford - M. Smart - J. Brown - J. Tatum - ...  1610612738   
4  P. Beverley - D. Russell - K. Towns - J. Vande...  1610612750   

  TEAM_ABBREVIATION  GP   W   L  W_PCT    MIN  ...  MIN_RANK  EFG_PCT_RANK  \
0               DEN  41  26  15  0.634  761.0  ...         1          5593   
1               PHX  38  31   7  0.816  754.0  ...         2       

In [6]:
def parse_player_ids(group_id):

    return [
        int(x)
        for x in group_id.split("-")
        if x
    ]

In [7]:
def weighted_average(df):

    if len(df) == 0:
        return pd.Series(
            {x: np.nan for x in FOUR_FACTORS}
        )

    return pd.Series({
        col: np.average(
            df[col],
            weights=df["MIN"]
        )
        for col in FOUR_FACTORS
    })

In [8]:
def build_player_four_factors(lineups):

    rows = []

    for team_id, team_df in lineups.groupby("TEAM_ID"):

        players = set(
            p
            for lineup in team_df["PLAYER_IDS"]
            for p in lineup
        )

        for player_id in players:

            on = team_df[
                team_df["PLAYER_IDS"]
                .apply(lambda x: player_id in x)
            ]

            off = team_df[
                team_df["PLAYER_IDS"]
                .apply(lambda x: player_id not in x)
            ]

            # skip players with no off court minutes
            if len(on) == 0 or len(off) == 0:
                continue

            on_stats = weighted_average(on)
            off_stats = weighted_average(off)

            row = {
                "PLAYER_ID": player_id,
                "TEAM_ID": team_id
            }

            for stat in FOUR_FACTORS:

                row[f"{stat}_ON"] = on_stats[stat]
                row[f"{stat}_OFF"] = off_stats[stat]

                row[f"{stat}_DIFF"] = (
                    on_stats[stat]
                    -
                    off_stats[stat]
                )
            
            row["MIN_ON"] = on["MIN"].sum()
            row["MIN_OFF"] = off["MIN"].sum()

            rows.append(row)

    return pd.DataFrame(rows)

In [12]:
seasons = [
    "2021-22",
    "2022-23",
    "2023-24",
    "2024-25",
    "2025-26"
]


all_features = []


for season in seasons:

    print("Starting", season)

    lineups = get_lineups(season)

    lineups["PLAYER_IDS"] = (
        lineups["GROUP_ID"]
        .apply(parse_player_ids)
    )

    features = build_player_four_factors(lineups)

    features["SEASON"] = season

    all_features.append(features)

    time.sleep(2)


four_factor_df = pd.concat(
    all_features,
    ignore_index=True
)

Starting 2021-22
Loading cached: 2021-22
Starting 2022-23
Saved 2022-23
Starting 2023-24
Saved 2023-24
Starting 2024-25
Saved 2024-25
Starting 2025-26
Saved 2025-26


In [13]:
four_factor_df = four_factor_df[
    four_factor_df["MIN_ON"] + four_factor_df["MIN_OFF"] > 300
]

In [14]:
four_factor_df = four_factor_df[
    four_factor_df["MIN_ON"] > 500
]

In [15]:
from nba_api.stats.endpoints import TeamPlayerOnOffDetails


def get_on_off_target(season):

    rows = []

    for team in teams.get_teams():

        print(team["full_name"])

        data = TeamPlayerOnOffDetails(
            team_id=team["id"],
            season=season,
            measure_type_detailed_defense="Advanced",
            per_mode_detailed="Per100Possessions"
        )

        on = data.get_data_frames()[1]
        off = data.get_data_frames()[2]


        on = on[[
            "VS_PLAYER_ID",
            "VS_PLAYER_NAME",
            "NET_RATING"
        ]]

        off = off[[
            "VS_PLAYER_ID",
            "NET_RATING"
        ]]


        df = on.merge(
            off,
            on="VS_PLAYER_ID",
            suffixes=("_ON", "_OFF")
        )


        df["TEAM_ID"] = team["id"]

        df["ON_OFF_NET"] = (
            df["NET_RATING_ON"]
            -
            df["NET_RATING_OFF"]
        )

        rows.append(df)


        time.sleep(1)


    return pd.concat(rows, ignore_index=True)

In [16]:
target_rows = []

for season in seasons:

    target = get_on_off_target(season)

    target["SEASON"] = season

    target_rows.append(target)


on_off_target = pd.concat(
    target_rows,
    ignore_index=True
)

Atlanta Hawks
Boston Celtics
Cleveland Cavaliers
New Orleans Pelicans
Chicago Bulls
Dallas Mavericks
Denver Nuggets
Golden State Warriors
Houston Rockets
Los Angeles Clippers
Los Angeles Lakers
Miami Heat
Milwaukee Bucks
Minnesota Timberwolves
Brooklyn Nets
New York Knicks
Orlando Magic
Indiana Pacers
Philadelphia 76ers
Phoenix Suns
Portland Trail Blazers
Sacramento Kings
San Antonio Spurs
Oklahoma City Thunder
Toronto Raptors
Utah Jazz
Memphis Grizzlies
Washington Wizards
Detroit Pistons
Charlotte Hornets
Atlanta Hawks
Boston Celtics
Cleveland Cavaliers
New Orleans Pelicans
Chicago Bulls
Dallas Mavericks
Denver Nuggets
Golden State Warriors
Houston Rockets
Los Angeles Clippers
Los Angeles Lakers
Miami Heat
Milwaukee Bucks
Minnesota Timberwolves
Brooklyn Nets
New York Knicks
Orlando Magic
Indiana Pacers
Philadelphia 76ers
Phoenix Suns
Portland Trail Blazers
Sacramento Kings
San Antonio Spurs
Oklahoma City Thunder
Toronto Raptors
Utah Jazz
Memphis Grizzlies
Washington Wizards
Detroit Pi

In [17]:
on_off_target

,VS_PLAYER_ID,VS_PLAYER_NAME,NET_RATING_ON,NET_RATING_OFF,TEAM_ID,ON_OFF_NET,SEASON
0,1627760,"Barber, Cat",34.6,-11.5,1610612737,46.1,2021-22
1,203992,"Bogdanović, Bogdan",4.5,-1.4,1610612737,5.9,2021-22
2,1630602,"Brown Jr., Chaundee",-10.9,3.1,1610612737,-14.0,2021-22
3,203991,"Capela, Clint",3.3,-0.3,1610612737,3.6,2021-22
4,1628381,"Collins, John",0.8,1.8,1610612737,-1.0,2021-22
...,...,...,...,...,...,...,...
3291,1629012,"Sexton, Collin",-0.5,4.1,1610612766,-4.6,2025-26
3292,1642354,"Simpson, KJ",-6.9,3.1,1610612766,-10.0,2025-26
3293,1630214,"Tillman, Xavier",4.2,10.5,1610612766,-6.3,2025-26
3294,1629632,"White, Coby",10.9,9.8,1610612766,1.1,2025-26


In [18]:
on_off_target = on_off_target.rename(
    columns={
        "VS_PLAYER_ID": "PLAYER_ID",
        "VS_PLAYER_NAME": "PLAYER_NAME"
    }
)

on_off_target = on_off_target.groupby(
    ["PLAYER_ID","SEASON"],
    as_index=False
).agg({
    "ON_OFF_NET":"mean"
})

In [19]:
four_factor_df

,PLAYER_ID,TEAM_ID,EFG_PCT_ON,EFG_PCT_OFF,EFG_PCT_DIFF,FTA_RATE_ON,FTA_RATE_OFF,FTA_RATE_DIFF,TM_TOV_PCT_ON,TM_TOV_PCT_OFF,...,OPP_FTA_RATE_DIFF,OPP_TOV_PCT_ON,OPP_TOV_PCT_OFF,OPP_TOV_PCT_DIFF,OPP_OREB_PCT_ON,OPP_OREB_PCT_OFF,OPP_OREB_PCT_DIFF,MIN_ON,MIN_OFF,SEASON
0,201568,1610612737,0.564719,0.540972,0.023746,0.258369,0.227632,0.030737,0.099157,0.118972,...,0.018147,0.124415,0.123692,0.000722,0.276330,0.264390,0.011940,1254.0,1586.0,2021-22
1,1629027,1610612737,0.557585,0.530894,0.026691,0.247377,0.220489,0.026888,0.110551,0.109120,...,0.023674,0.123713,0.125014,-0.001301,0.264657,0.286460,-0.021803,2188.0,652.0,2021-22
5,1626153,1610612737,0.537530,0.557367,-0.019837,0.254525,0.235553,0.018972,0.101418,0.113958,...,0.003843,0.137517,0.118281,0.019235,0.292954,0.259780,0.033174,846.0,1994.0,2021-22
7,1628989,1610612737,0.555205,0.545059,0.010146,0.225525,0.267974,-0.042449,0.110279,0.110126,...,0.004731,0.121367,0.128525,-0.007158,0.253721,0.296880,-0.043159,1791.0,1049.0,2021-22
8,1630168,1610612737,0.568463,0.546669,0.021794,0.272792,0.232310,0.040482,0.107845,0.110892,...,0.026268,0.139821,0.119560,0.020261,0.312061,0.257723,0.054338,624.0,2216.0,2021-22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2715,1628970,1610612766,0.575969,0.567814,0.008156,0.222049,0.251620,-0.029571,0.136629,0.169823,...,-0.046820,0.113630,0.102853,0.010777,0.263077,0.281417,-0.018340,1665.0,429.0,2025-26
2716,1641706,1610612766,0.583567,0.559576,0.023991,0.234697,0.217640,0.017057,0.145324,0.140422,...,-0.029549,0.111689,0.110999,0.000690,0.259337,0.278744,-0.019407,1285.0,809.0,2025-26
2720,1631217,1610612766,0.581930,0.564193,0.017737,0.232869,0.221802,0.011067,0.136724,0.152309,...,-0.015373,0.118674,0.101820,0.016854,0.270204,0.262374,0.007830,1193.0,901.0,2025-26
2722,1630163,1610612766,0.586782,0.553482,0.033301,0.208694,0.260479,-0.051785,0.132696,0.161329,...,-0.002365,0.115062,0.105353,0.009709,0.269577,0.262262,0.007314,1309.0,785.0,2025-26


In [20]:
on_off_target.columns

Index(['PLAYER_ID', 'SEASON', 'ON_OFF_NET'], dtype='object')

In [80]:
model_df = four_factor_df.merge(
    on_off_target,
    on=[
        "PLAYER_ID",
        "SEASON"
    ],
    how="inner"
)

In [22]:
model_df.columns

Index(['PLAYER_ID', 'TEAM_ID', 'EFG_PCT_ON', 'EFG_PCT_OFF', 'EFG_PCT_DIFF',
       'FTA_RATE_ON', 'FTA_RATE_OFF', 'FTA_RATE_DIFF', 'TM_TOV_PCT_ON',
       'TM_TOV_PCT_OFF', 'TM_TOV_PCT_DIFF', 'OREB_PCT_ON', 'OREB_PCT_OFF',
       'OREB_PCT_DIFF', 'OPP_EFG_PCT_ON', 'OPP_EFG_PCT_OFF',
       'OPP_EFG_PCT_DIFF', 'OPP_FTA_RATE_ON', 'OPP_FTA_RATE_OFF',
       'OPP_FTA_RATE_DIFF', 'OPP_TOV_PCT_ON', 'OPP_TOV_PCT_OFF',
       'OPP_TOV_PCT_DIFF', 'OPP_OREB_PCT_ON', 'OPP_OREB_PCT_OFF',
       'OPP_OREB_PCT_DIFF', 'MIN_ON', 'MIN_OFF', 'SEASON', 'ON_OFF_NET'],
      dtype='object')

In [81]:
df = model_df.copy()

# keep identifiers
id_cols = [
    "PLAYER_ID",
    "PLAYER_NAME",
    "TEAM_ID",
    "SEASON"
]


# minutes played on/off
df["MIN_TOTAL"] = (
    df["MIN_ON"] +
    df["MIN_OFF"]
)


# normalize how much evidence we have
df["ON_SHARE"] = (
    df["MIN_ON"] /
    df["MIN_TOTAL"]
)


df["OFF_SHARE"] = (
    df["MIN_OFF"] /
    df["MIN_TOTAL"]
)


print(df.shape)

(1260, 33)


In [90]:
model_df = df

In [91]:
weighted_cols = [
    "EFG_PCT_ON",
    "EFG_PCT_OFF",
    "EFG_PCT_DIFF",
    "FTA_RATE_ON",
    "FTA_RATE_OFF",
    "FTA_RATE_DIFF",
    "TM_TOV_PCT_ON",
    "TM_TOV_PCT_OFF",
    "TM_TOV_PCT_DIFF",
    "OREB_PCT_ON",
    "OREB_PCT_OFF",
    "OREB_PCT_DIFF",
    "OPP_EFG_PCT_ON",
    "OPP_EFG_PCT_OFF",
    "OPP_EFG_PCT_DIFF",
    "OPP_FTA_RATE_ON",
    "OPP_FTA_RATE_OFF",
    "OPP_FTA_RATE_DIFF",
    "OPP_TOV_PCT_ON",
    "OPP_TOV_PCT_OFF",
    "OPP_TOV_PCT_DIFF",
    "OPP_OREB_PCT_ON",
    "OPP_OREB_PCT_OFF",
    "OPP_OREB_PCT_DIFF",
    "ON_OFF_NET",
]

In [92]:
def merge_traded_players(df):
    rows = []

    for (player_id, season), group in df.groupby(["PLAYER_ID", "SEASON"]):
        w = group["MIN_ON"]

        row = {
            "PLAYER_ID": player_id,
            "SEASON": season,
            "MIN_ON": group["MIN_ON"].sum(),
            "MIN_OFF": group["MIN_OFF"].sum(),
        }

        row["MIN_TOTAL"] = row["MIN_ON"] + row["MIN_OFF"]
        row["ON_SHARE"] = row["MIN_ON"] / row["MIN_TOTAL"]

        for col in weighted_cols:
            row[col] = np.average(group[col], weights=w)

        rows.append(row)

    return pd.DataFrame(rows)

In [93]:
model_df = merge_traded_players(model_df)

### Model Creation and Training

In [94]:
# train test split
train_df = model_df[
    model_df["SEASON"].isin(
        ["2021-22", "2022-23", "2023-24", "2024-25"]
    )
].copy()

test_df = model_df[
    model_df["SEASON"] == "2025-26"
].copy()

In [95]:
# filtering out noisy samples
train_df = train_df[
    train_df["MIN_TOTAL"] > 1000
]

test_df = test_df[
    test_df["MIN_TOTAL"] > 1000
]

In [96]:
features = [

    # offensive four factors
    # "EFG_PCT_ON",
    # "EFG_PCT_OFF",
    "EFG_PCT_DIFF",

    # "FTA_RATE_ON",
    # "FTA_RATE_OFF",
    "FTA_RATE_DIFF",

    # "TM_TOV_PCT_ON",
    # "TM_TOV_PCT_OFF",
    "TM_TOV_PCT_DIFF",

    # "OREB_PCT_ON",
    # "OREB_PCT_OFF",
    "OREB_PCT_DIFF",


    # defensive four factors
    # "OPP_EFG_PCT_ON",
    # "OPP_EFG_PCT_OFF",
    "OPP_EFG_PCT_DIFF",

    # "OPP_FTA_RATE_ON",
    # "OPP_FTA_RATE_OFF",
    "OPP_FTA_RATE_DIFF",

    # "OPP_TOV_PCT_ON",
    # "OPP_TOV_PCT_OFF",
    "OPP_TOV_PCT_DIFF",

    # "OPP_OREB_PCT_ON",
    # "OPP_OREB_PCT_OFF",
    "OPP_OREB_PCT_DIFF",


    # sample size controls
    # "MIN_ON",
    # "MIN_OFF",
    "MIN_TOTAL",
    "ON_SHARE"

]

In [97]:
X_train = train_df[features]
y_train = train_df["ON_OFF_NET"]

X_test = test_df[features]
y_test = test_df["ON_OFF_NET"]

In [98]:
season_order = {
    "2021-22":0,
    "2022-23":1,
    "2023-24":2,
    "2024-25":3,
    "2025-26":4
}

model_df["SEASON_NUM"] = (
    model_df["SEASON"]
    .map(season_order)
)

model_df = model_df.sort_values("SEASON_NUM")



In [99]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
import numpy as np


tscv = TimeSeriesSplit(
    n_splits=3
)

X = X_train
y = y_train

alphas = [
    .01,
    .1,
    1,
    10,
    50,
    100
]


for alpha in alphas:

    model = Pipeline([
        (
            "scale",
            StandardScaler()
        ),
        (
            "ridge",
            Ridge(alpha=alpha)
        )
    ])


    scores = []


    for train_idx, val_idx in tscv.split(X):

        X_train = X.iloc[train_idx]
        X_val = X.iloc[val_idx]

        y_train = y.iloc[train_idx]
        y_val = y.iloc[val_idx]


        model.fit(
            X_train,
            y_train
        )


        pred = model.predict(
            X_val
        )


        mae = mean_absolute_error(
            y_val,
            pred
        )

        scores.append(mae)


    print(
        alpha,
        np.mean(scores)
    )

0.01 2.648295530622957
0.1 2.648171544348084
1 2.646963104873095
10 2.638575100348309
50 2.635438681945047
100 2.6805319591613856


In [100]:
model = Pipeline([
        (
            "scale",
            StandardScaler()
        ),
        (
            "ridge",
            Ridge(alpha=50)
        )
    ])

In [101]:
model.fit(X_train,y_train)

,steps,"[('scale', ...), ('ridge', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,alpha,50
,fit_intercept,True
,copy_X,True
,max_iter,None


In [102]:
predictions = model.predict(X_test)
results_df = test_df.assign(Predicted_Value=predictions)


In [104]:
from nba_api.stats.endpoints import LeagueDashPlayerStats

seasons = [
    "2021-22",
    "2022-23",
    "2023-24",
    "2024-25",
    "2025-26"
]

name_map = {}
# team_map = {}

for season in seasons:
    df = LeagueDashPlayerStats(
        season=season,
        per_mode_detailed="PerGame"
    ).get_data_frames()[0]

    name_season_map = (
        df[["PLAYER_ID", "PLAYER_NAME"]]
        .drop_duplicates()
        .set_index("PLAYER_ID")["PLAYER_NAME"]
        .to_dict()
    )

    # team_season_map = (
    #     df[["TEAM_ID", "TEAM_ABBREVIATION"]]
    #     .drop_duplicates()
    #     .set_index("TEAM_ID")["TEAM_ABBREVIATION"]
    #     .to_dict()
    # )

    name_map.update(name_season_map)
    # team_map.update(team_season_map)

model_df["PLAYER_NAME"] = model_df["PLAYER_ID"].map(name_map)
# model_df["TEAM_ABBREVIATION"] = model_df["TEAM_ID"].map(team_map)


In [106]:
results_df["PLAYER_NAME"] = results_df["PLAYER_ID"].map(name_map)
# results_df["TEAM_ABBREVIATION"] = results_df["TEAM_ID"].map(team_map)

In [107]:
# check for missing players
missing = results_df[results_df["PLAYER_NAME"].isna()]
print(missing[["PLAYER_ID"]])

Empty DataFrame
Columns: [PLAYER_ID]
Index: []


### Results

In [ ]:
# checking an individual player's features + result
results_df.sort_values(by='Predicted_Value', ascending=False).iloc[7]

PLAYER_ID                  203999
TEAM_ID                1610612743
EFG_PCT_ON               0.602808
EFG_PCT_OFF              0.562886
EFG_PCT_DIFF             0.039922
FTA_RATE_ON              0.315217
FTA_RATE_OFF             0.249276
FTA_RATE_DIFF            0.065941
TM_TOV_PCT_ON            0.121476
TM_TOV_PCT_OFF           0.128465
TM_TOV_PCT_DIFF         -0.006989
OREB_PCT_ON              0.291801
OREB_PCT_OFF             0.262555
OREB_PCT_DIFF            0.029246
OPP_EFG_PCT_ON           0.558035
OPP_EFG_PCT_OFF          0.543172
OPP_EFG_PCT_DIFF         0.014863
OPP_FTA_RATE_ON           0.23808
OPP_FTA_RATE_OFF         0.270942
OPP_FTA_RATE_DIFF       -0.032862
OPP_TOV_PCT_ON           0.113918
OPP_TOV_PCT_OFF          0.096106
OPP_TOV_PCT_DIFF         0.017812
OPP_OREB_PCT_ON          0.260968
OPP_OREB_PCT_OFF         0.291317
OPP_OREB_PCT_DIFF       -0.030348
MIN_ON                     1866.0
MIN_OFF                     849.0
SEASON                    2025-26
ON_OFF_NET    

In [108]:
mae = mean_absolute_error(
            y_test,
            predictions
        )

In [109]:
mae

2.911557136109441

In [111]:
results_df = results_df[['PLAYER_NAME', 'Predicted_Value']]
df_sorted = results_df.sort_values(by='Predicted_Value', ascending=False)
df_sorted

,PLAYER_NAME,Predicted_Value
1130,Victor Wembanyama,13.419358
1058,Chet Holmgren,12.406975
729,Nickeil Alexander-Walker,12.373288
1231,Cedric Coward,11.802387
213,Giannis Antetokounmpo,11.685183
...,...,...
711,De'Andre Hunter,-7.595128
1177,Ronald Holland II,-8.597408
875,Isaiah Stewart,-9.057905
99,Jonas Valanciunas,-10.059722


In [112]:
# exporting results
df_sorted.to_csv("outputs/four_factor_model_results_v4.csv", index=False)

### Feature Importances

In [43]:
regression_pipe_coefs = pd.DataFrame(
    data=model.named_steps["ridge"].coef_, index=features, columns=["Coefficient"]
)

In [ ]:
regression_pipe_coefs.sort_values(by="Coefficient", ascending=False).head(80)

,Coefficient
EFG_PCT_DIFF,2.586925
OREB_PCT_DIFF,1.445162
ON_SHARE,1.155937
OPP_TOV_PCT_DIFF,1.142449
FTA_RATE_DIFF,0.726002
MIN_TOTAL,-0.082245
OPP_FTA_RATE_DIFF,-0.948292
TM_TOV_PCT_DIFF,-0.959765
OPP_OREB_PCT_DIFF,-1.186652
OPP_EFG_PCT_DIFF,-2.355123
